# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mariamemad975/FlyRank_ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — “What Predicts Health?”

The Random Forest reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the strongest predictors of Health Score. However, these variables are direct components of the Health Score formula itself: Impressions (30), Position (30), CTR (20), and Scroll Depth (20).

Methodological question: Does predicting a composite score from its own ingredients provide information beyond confirming the construction formula? The feature importance may be better interpreted as construction weight rather than independent predictive importance.

Finding 2 — “What Predicts Growth?”

The Logistic Regression reports 71% holdout accuracy, with Content Age, Days Since Update, and Days Visible as the strongest signals. Trend Direction, however, is defined from the 30-day vs. previous-30-day impression change.

Methodological question: Were any input features derived from or closely aligned with this same window, making them near-restatements of the label? Also, was the holdout split random or time-aware? A random split could inflate performance through temporal overlap or near-duplicate observations.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model compared Random Forest and Gradient Boosting with a Logistic Regression baseline for CTR/engagement decay. Permutation importance showed that impressions_last_30d and impressions_prev_30d accounted for ~86% of total importance, suggesting potential target leakage rather than a generalizable pattern.

Split choice: I used client_id-grouped validation. Since the dataset has no exposed date/timestamp field, a random split could place the same client’s pages in both train and test, allowing the model to learn client-specific patterns. GroupKFold by client_id prevents this and provides a more honest estimate of generalization across clients.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip install -q duckdb pandas scikit-learn matplotlib

In [32]:
!git clone https://github.com/mariamemad975/FlyRank_ML.git

Cloning into 'FlyRank_ML'...
remote: Enumerating objects: 254, done.
remote: Counting objects: 100% (254/254), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 254 (delta 132), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (254/254), 2.02 MiB | 9.36 MiB/s, done.
Resolving deltas: 100% (132/132), done.


In [33]:
%cd FlyRank_ML

/content/FlyRank_ML/FlyRank_ML/FlyRank_ML


In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold,train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, average_precision_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Dataset shape:", df.shape)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

drop_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

feature_cols = [c for c in df.columns if c not in drop_cols]

numeric_features = (
    df[feature_cols]
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

print(f"{len(numeric_features)} numeric features:", numeric_features)

X = df[numeric_features].fillna(0)
y = df["is_declining_label"]

groups = df["client_id"]

Dataset shape: (30000, 44)
29 numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y)

model_before = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=20,
    random_state=42)

model_before.fit(X_train, y_train)

proba_before = model_before.predict_proba(X_test)[:, 1]
pred_before = model_before.predict(X_test)

metrics_before = {
    "split": "Random split (before)",
    "ROC-AUC": roc_auc_score(y_test, proba_before),
    "Average Precision": average_precision_score(y_test, proba_before),
    "Precision": precision_score(y_test, pred_before),
    "Recall": recall_score(y_test, pred_before),}

print(metrics_before)

{'split': 'Random split (before)', 'ROC-AUC': np.float64(0.820080040320054), 'Average Precision': np.float64(0.8301650884731916), 'Precision': 0.7192168985059247, 'Recall': 0.8585485854858549}


In [36]:
# Grouped cross-validation by client_id (after)
gkf = GroupKFold(n_splits=5)
fold_metrics = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups=groups), start=1
):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_leaf=20,
        random_state=42)

    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)

    fold_metrics.append({
        "fold": fold,
        "roc_auc": roc_auc_score(y_test, proba),
        "avg_precision": average_precision_score(y_test, proba),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),})

fold_df = pd.DataFrame(fold_metrics)

print("Fold results:")
print(fold_df)

metrics_after = {
    "split": "GroupKFold by client_id (after)",
    "roc_auc": fold_df["roc_auc"].mean(),
    "avg_precision": fold_df["avg_precision"].mean(),
    "precision": fold_df["precision"].mean(),
    "recall": fold_df["recall"].mean(),}

print("\nMean across folds:")
print(metrics_after)

Fold results:
   fold   roc_auc  avg_precision  precision    recall
0     1  0.710903       0.702612   0.623410  0.699272
1     2  0.739517       0.830231   0.708593  0.951879
2     3  0.851347       0.760175   0.570291  0.951443
3     4  0.752079       0.790256   0.762036  0.928232
4     5  0.799414       0.811969   0.738786  0.915577

Mean across folds:
{'split': 'GroupKFold by client_id (after)', 'roc_auc': np.float64(0.7706520507715483), 'avg_precision': np.float64(0.7790486434970278), 'precision': np.float64(0.6806233399815615), 'recall': np.float64(0.889280616857417)}


In [37]:
metrics_before_standardized = {
    'split': metrics_before['split'],
    'roc_auc': metrics_before['ROC-AUC'],
    'avg_precision': metrics_before['Average Precision'],
    'precision': metrics_before['Precision'],
    'recall': metrics_before['Recall'],
}

comparison = pd.DataFrame([metrics_before_standardized, metrics_after]).set_index("split")
print(comparison)
print("\nGap (before - after):")
print(comparison.loc["Random split (before)"] - comparison.loc["GroupKFold by client_id (after)"])

                                  roc_auc  avg_precision  precision    recall
split                                                                        
Random split (before)            0.820080       0.830165   0.719217  0.858549
GroupKFold by client_id (after)  0.770652       0.779049   0.680623  0.889281

Gap (before - after):
roc_auc          0.049428
avg_precision    0.051116
precision        0.038594
recall          -0.030732
dtype: float64


The random-to-grouped split gap is real but moderate: ROC-AUC falls from 0.820 to 0.771, while precision drops from 0.72 to 0.68. Recall slightly improves (0.86 → 0.89), so performance does not collapse. However, the consistent decline in ROC-AUC, average precision, and precision indicates that the random split likely benefited from client-specific patterns appearing in both training and test sets.

Therefore, I would report the GroupKFold results as the honest performance: ROC-AUC ≈ 0.77, precision ≈ 0.68, and recall ≈ 0.89.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [38]:
# 1. Verify that label-source columns are excluded from the features.
label_source_cols = ["trend_direction", "trend_pct"]

leaked = [col for col in numeric_features if col in label_source_cols]
assert not leaked, f"Direct label leakage detected: {leaked}"

print("No direct label-source columns are included in the feature set.")

# 2. Check for features with unusually high correlation with the label.
corrs = (
    X.assign(label=y)
     .corr()["label"]
     .drop("label")
     .sort_values(key=abs, ascending=False))

print("\nTop feature correlations with the label:")
print(corrs.head(10).round(3))

No direct label-source columns are included in the feature set.

Top feature correlations with the label:
days_with_impressions     0.190
content_age_days         -0.164
age_tier_order           -0.156
word_count                0.119
char_count                0.108
impressions_last_30d     -0.094
days_since_last_update    0.081
clicks_last_30d          -0.072
sessions_last_30d        -0.064
ctr                      -0.062
Name: label, dtype: float64


In [39]:
# 3. Feature-importance concentration check
importances = (
    pd.Series(model_before.feature_importances_, index=numeric_features)
    .sort_values(ascending=False)
)

print("Top feature importances (random-split model):")
print(importances.head(10).round(3))

top2_share = importances.head(2).sum() / importances.sum()
print(f"\nTop-2 features account for {top2_share:.1%} of total importance.")

if top2_share > 0.70:
    print("\nFLAG: Strong importance concentration in the top two features.")
    print("The leading features appear to capture recent-vs-previous impression changes,")
    print("which closely mirror the information used to define trend_direction.")
    print("Although the label-source columns are excluded, these features may act as")
    print("near-proxies for the target rather than independent CTR/engagement signals.")

Top feature importances (random-split model):
impressions_prev_30d     0.354
impressions_90d          0.103
impressions_last_30d     0.090
days_with_impressions    0.086
avg_position             0.072
content_age_days         0.064
age_tier_order           0.036
clicks_last_30d          0.032
word_count               0.029
sessions_last_30d        0.025
dtype: float64

Top-2 features account for 45.7% of total importance.


The retrained feature set shows less importance concentration than the original ML-08 run. The top two features, impressions_prev_30d and impressions_90d, account for 45.7% of total importance versus ~86% originally, likely due to removing categorical tier features and adding impressions_90d.

Feature–label correlations remain modest, with days_with_impressions having the highest correlation (r = 0.19), so there is no clear direct leakage. However, impressions_prev_30d and impressions_last_30d still represent an impression-trend signal closely related to the trend_direction label. Therefore, their contribution to model performance should be interpreted with caution, similar to the concern raised in Finding 2.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
claims = [
    { "original": "The model predicts which pages will decline.",
        "rewritten": (
            "The model's score is associated with observed decline under the "
            "client-grouped evaluation (mean ROC-AUC of {:.2f} across 5 held-out "
            "client folds). It should be used as a decision-support signal for "
            "prioritizing review, not as a certain prediction."
        ).format(metrics_after["roc_auc"]),},
    {
        "original": "Accuracy of 82.1% proves the model works well.",
        "rewritten": (
            "Under the client-grouped evaluation, the model achieved {:.2f} "
            "precision and {:.2f} recall. These results are lower than the "
            "same-client random split, so the split gap should be disclosed "
            "when reporting model performance.").format(
            metrics_after["precision"],
            metrics_after["recall"]),},]

for claim in claims:
    print("ORIGINAL: ", claim["original"])
    print("REWRITTEN:", claim["rewritten"])
    print()

ORIGINAL:  The model predicts which pages will decline.
REWRITTEN: The model's score is associated with observed decline under the client-grouped evaluation (mean ROC-AUC of 0.77 across 5 held-out client folds). It should be used as a decision-support signal for prioritizing review, not as a certain prediction.

ORIGINAL:  Accuracy of 82.1% proves the model works well.
REWRITTEN: Under the client-grouped evaluation, the model achieved 0.68 precision and 0.89 recall. These results are lower than the same-client random split, so the split gap should be disclosed when reporting model performance.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.